# Usami Singing Pipeline (Estilo Neuro-sama) — Treino & Conversao RVC v2 (Applio)

Este notebook treina o modelo **RVC v2 (`40 kHz` + `RMVPE` + `ContentVec`)** da **Usami** usando a **GPU T4 gratuita do Google Colab** e ja inclui uma celula pronta para converter qualquer guia vocal (exportada do **OpenUTAU / DiffSinger** ou acapella) na voz cantada da Usami.

### Antes de comecar:
1. No menu superior do Colab, va em **Ambiente de execucao -> Alterar o tipo de ambiente de execucao -> GPU T4**.
2. Tenha em maos o arquivo `usami_rvc_dataset_30min.zip` gerado localmente em `usami/data/rvc/usami_rvc_dataset_30min.zip`.

In [ ]:
# @title 1. Instalar Applio + Baixar Modelos Base (RMVPE, ContentVec e HiFi-GAN 40k)
import os
import torch

assert torch.cuda.is_available(), 'Ative a GPU T4 em Ambiente de execucao -> Alterar o tipo de ambiente de execucao!'
print('GPU detectada:', torch.cuda.get_device_name(0))

if not os.path.exists('/content/Applio'):
    !git clone --depth 1 https://github.com/IAHispano/Applio.git /content/Applio
%cd /content/Applio
!pip install -q -r requirements.txt
!python core.py prerequisites --pretraineds-hifigan --models --no-exe
assert os.path.exists('/content/Applio/rvc/models/predictors/rmvpe.pt'), 'Falha ao baixar rmvpe.pt!'
print('Applio e modelos base (rmvpe.pt + contentvec + HiFi-GAN) prontos!')

In [ ]:
# @title 2. Enviar o Dataset Limpo da Usami (`usami_rvc_dataset_30min.zip`)
import os, zipfile
from google.colab import files

dataset_dir = '/content/Applio/assets/datasets/Usami'
os.makedirs(dataset_dir, exist_ok=True)

print('Selecione o arquivo usami_rvc_dataset_30min.zip:')
uploaded = files.upload()
for fname in uploaded.keys():
    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall(dataset_dir)

wavs = [f for f in os.listdir(dataset_dir) if f.lower().endswith('.wav')]
print(f'Dataset descompactado: {len(wavs)} clipes WAV prontos em {dataset_dir}')

In [ ]:
# @title 3. Preprocessar, Extrair Pitch (RMVPE), Gerar Index e Treinar Modelo `Usami` (~20 min na T4)
import glob, os, zipfile
from google.colab import files
%cd /content/Applio

TOTAL_EPOCHS = 200  # @param {type:"slider", min:100, max:350, step:25}
BATCH_SIZE = 8      # @param {type:"slider", min:4, max:16, step:2}

# Garante que rmvpe.pt e contentvec estao baixados mesmo se a Celula 1 nao baixou antes
if not os.path.exists('/content/Applio/rvc/models/predictors/rmvpe.pt'):
    !python core.py prerequisites --pretraineds-hifigan --models --no-exe

# Pula preprocess caso os 744 arquivos ja tenham sido cortados na sessao atual
sliced_dir = '/content/Applio/logs/Usami/sliced_audios'
if not os.path.isdir(sliced_dir) or len(os.listdir(sliced_dir)) == 0:
    !python core.py preprocess --model-name Usami --dataset-path "/content/Applio/assets/datasets/Usami" --sample-rate 40000 --cpu-cores 2
else:
    print('Preprocess ja concluido nesta sessao; indo direto para extracao RMVPE!')

!python core.py extract --model-name Usami --sample-rate 40000 --cpu-cores 2 --gpu 0 --f0-method rmvpe --embedder-model contentvec
!python core.py index --model-name Usami --index-algorithm Auto
!python core.py train --model-name Usami --sample-rate 40000 --gpu 0 --batch-size {BATCH_SIZE} --save-every-epoch 25 --total-epoch {TOTAL_EPOCHS} --save-only-latest --save-every-weights

# Filtra apenas pesos leves de inferencia (~55 MB), ignorando checkpoints gigantes G_*.pth e D_*.pth
infer_pths = sorted(
    [p for p in glob.glob('/content/Applio/logs/Usami/*.pth') if not os.path.basename(p).startswith(('G_', 'D_'))],
    key=os.path.getmtime,
)
idxs = sorted(glob.glob('/content/Applio/logs/Usami/*.index'), key=os.path.getmtime)
print('Pesos de inferencia gerados:', infer_pths)
print('Indices gerados:', idxs)

if infer_pths:
    bundle_path = '/content/Usami_RVC_v2_Model.zip'
    with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(infer_pths[-1], arcname='Usami.pth')
        if idxs:
            zf.write(idxs[-1], arcname='Usami.index')
    print('Baixando modelo final Usami_RVC_v2_Model.zip...')
    files.download(bundle_path)

In [ ]:
# @title 4. Converter Guia Cantada (OpenUTAU DiffSinger / Acapella) na Voz da Usami!
import glob, os
from google.colab import files
%cd /content/Applio

PITCH_SHIFT_SEMITONES = 0  # @param {type:"slider", min:-24, max:24, step:1}
INDEX_RATE = 0.72          # @param {type:"slider", min:0.0, max:1.0, step:0.05}
PROTECT_CONSONANTS = 0.33  # @param {type:"slider", min:0.0, max:0.5, step:0.05}

infer_pths = sorted(
    [p for p in glob.glob('/content/Applio/logs/Usami/*.pth') if not os.path.basename(p).startswith(('G_', 'D_'))],
    key=os.path.getmtime,
)
idxs = sorted(glob.glob('/content/Applio/logs/Usami/*.index'), key=os.path.getmtime)
assert infer_pths, 'Treine (ou coloque Usami.pth em /content/Applio/logs/Usami/) antes de converter!'
best_pth = infer_pths[-1]
best_idx = idxs[-1] if idxs else ''
print('Usando modelo:', best_pth, '| index:', best_idx)

print('Suba o arquivo .wav cantado (ex.: exportado do OpenUTAU DiffSinger):')
up_guide = files.upload()
for guide_name in up_guide.keys():
    in_wav = os.path.abspath(guide_name)
    out_wav = f'/content/usami_singing_{os.path.splitext(guide_name)[0]}.wav'
    !python core.py infer --input-path "{in_wav}" --output-path "{out_wav}" --pth-path "{best_pth}" --index-path "{best_idx}" --f0-method rmvpe --pitch {PITCH_SHIFT_SEMITONES} --index-rate {INDEX_RATE} --protect {PROTECT_CONSONANTS} --export-format WAV
    if os.path.exists(out_wav):
        files.download(out_wav)